# Probe: clean-day expert residual DQA-MoX

- created_utc: 2026-05-11T15:17:49+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2/logs/27e_probe_clean_day_expert_anchor_r2_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27e_probe_clean_day_expert_anchor_r2.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 24a_client_dominant_soft_expand | 0.457000 | 0.258000 |
| 27a_soft_mixture_head_first_40_10 |  |  |
| 27b_localization_uncertainty_strict_then_open |  |  |
| 27b_probe_localization_uncertainty_r2 | 0.462000 | 0.260000 |
| 27c_probe_k6_night_tail_r2 | 0.459000 | 0.250000 |
| 27d_probe_teacher_residual_mixpl_r2 | 0.462000 | 0.259000 |

## Hypothesis

27dはtotal mAP50を0.462まで戻したが、residential_night mAP50:95を0.236→0.214に落とした。一方でday側とhighway_nightは微増し、pseudoGTの悪さはclient/domain依存に見える。FedLGMatch/URSDの示唆に寄せ、夜clientを短期probeから外してday clientだけをclean local expertとして使い、nightは強いserver teacher anchorで守る。平均化ではなく、clean domain residualをMoE headに吸わせる2-round probe。

## Paper Basis

- FedMoX/PSSFL: https://arxiv.org/abs/2508.16568
  FedMoX treats the practical setting as server labeled high-resolution data plus client unlabeled low-resolution data, and uses sparse MoE with a spatial router and Soft-Mixture to stabilize semi-supervised FL.
- Mixed Pseudo Labels: https://arxiv.org/abs/2312.07006
  MixPL shows that pseudo labels can amplify both a detector's strengths and weaknesses, especially missed detections for small and tail-category objects; this supports keeping pseudoGT as a weak residual/domain signal when recent probes drift below warmup.
- FedLGMatch: https://www.sciencedirect.com/science/article/pii/S0950705125006884
  FedLGMatch emphasizes joint local/global pseudo labeling in federated SSL. For DQA-MoX, this motivates using clean local clients as expert residuals while keeping the global teacher as the dominant fallback for noisy domains.
- URSD: https://www.sciencedirect.com/science/article/pii/S0957417425038965
  URSD argues that sample mining uncertainty and anchor assignment uncertainty can block SSOD gains; this supports pruning noisy night pseudoGT instead of forcing every client into every short probe.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2/logs/27e_probe_clean_day_expert_anchor_r2_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '4', '--top-k', '2', '--router-temperature', '1.05', '--router-balance-weight', '0.02', '--router-entropy-weight', '0.0004', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--clients', '0,2,4', '--warmup-epochs', '50', '--client-limit', '1600', '--client-sampling-ratio', '1.000', '--client-sampling-seed', '270411', '--phase1-rounds', '2', '--phase2-rounds', '0', '--phase1-train-scope', 'neck_head', '--phase1-repair-train-scope', 'neck_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00018', '--phase1-source-repeat', '2', '--phase1-pseudo-repeat', '2', '--phase1-loss-box', '0.00016', '--phase2-train-scope', 'all', '--phase2-repair-train-scope', 'all', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.00003', '--phase2-source-repeat', '2', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00003', '--server-repair-epochs', '1', '--server-repair-lr', '0.00012', '--server-repair-loss-box', '0.004', '--dqa-temperature', '0.85', '--dqa-uniform-mix', '0.12', '--dqa-classwise-blend', '0.25', '--dqa-stability-lambda', '0.55', '--dqa-server-anchor', '0.55', '--dqa-min-server-alpha', '0.50', '--dqa-residual-blend', '0.06', '--late-dqa-server-anchor', '0.50', '--late-dqa-min-server-alpha', '0.46', '--late-dqa-residual-blend', '0.06', '--curriculum-start-round', '3', '--expert-keep-fraction', '0.84', '--expert-max-class-fraction', '0.34', '--actual-max-class-fraction', '0.44', '--late-expert-keep-fraction', '0.90', '--late-expert-max-class-fraction', '0.38', '--late-actual-max-class-fraction', '0.50', '--min-score', '0.18', '--min-stability', '0.55', '--late-min-score', '0.14', '--late-min-stability', '0.44', '--max-boxes-per-image', '12', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


In [ ]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27e_probe_clean_day_expert_anchor_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)
